In [ ]:
# 1. Clona il repository

!git clone -b standard-gaussian-with-mask https://github.com/baddy2002/image-gs

%cd image-gs

Cloning into 'image-gs'...
remote: Enumerating objects: 1863, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 1863 (delta 111), reused 131 (delta 87), pack-reused 1682 (from 1)
Receiving objects: 100% (1863/1863), 8.73 MiB | 24.54 MiB/s, done.
Resolving deltas: 100% (1036/1036), done.
/content/image-gs/image-gs


In [1]:
# 3. Installazione dipendenze base da environment.yml
!pip install pytorch-msssim==1.0.0 torchmetrics==1.5.2 lpips==0.1.4 flip-evaluator plyfile

In [ ]:
# 4. Installiamo il gsplat locale
%cd /content/image-gs/gsplat
!pip install -e . -v
%cd ..

/content/image-gs/gsplat
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Obtaining file:///content/image-gs/gsplat
  Running command python setup.py egg_info
  running egg_info
  creating /tmp/pip-pip-egg-info-q12eg0zc/gsplat.egg-info
  writing /tmp/pip-pip-egg-info-q12eg0zc/gsplat.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-pip-egg-info-q12eg0zc/gsplat.egg-info/dependency_links.txt
  writing requirements to /tmp/pip-pip-egg-info-q12eg0zc/gsplat.egg-info/requires.txt
  writing top-level names to /tmp/pip-pip-egg-info-q12eg0zc/gsplat.egg-info/top_level.txt
  writing manifest file '/tmp/pip-pip-egg-info-q12eg0zc/gsplat.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-pip-egg-info-q12eg0zc/gsplat.egg-info/SOURCES.txt'
  writing manifest file '/tmp/pip-pip-egg-info-q12eg0zc/gsplat.egg-info/SOURCES.txt'
  Preparing metadata (setup.py) ... done
  Attempting uninstall: gsplat
    Found existing installation: gsplat 1.4.0
    Uninstalling gsp

In [9]:

# 6. Test finale
!python main.py --help

usage: main.py [-h] [--seed SEED] [--device DEVICE] [--eval]
               [--render_height RENDER_HEIGHT] [--quantize]
               [--pos_bits POS_BITS] [--scale_bits SCALE_BITS]
               [--rot_bits ROT_BITS] [--feat_bits FEAT_BITS]
               [--log_root LOG_ROOT] [--exp_name EXP_NAME]
               [--log_level LOG_LEVEL] [--save_image_format SAVE_IMAGE_FORMAT]
               [--save_plot_format SAVE_PLOT_FORMAT] [--vis_gaussians]
               [--save_image_steps SAVE_IMAGE_STEPS]
               [--save_ckpt_steps SAVE_CKPT_STEPS] [--eval_steps EVAL_STEPS]
               [--gamma GAMMA] [--data_root DATA_ROOT]
               [--input_path INPUT_PATH] [--downsample]
               [--downsample_ratio DOWNSAMPLE_RATIO]
               [--num_gaussians NUM_GAUSSIANS] [--init_scale INIT_SCALE]
               [--topk TOPK] [--disable_topk_norm] [--disable_inverse_scale]
               [--ckpt_file CKPT_FILE] [--disable_color_init]
               [--init_mode INIT_MODE] [

In [4]:
# Installa lo strumento di conversione
!apt-get install imagemagick -y

!ls -lh media/textures/castpol01.png

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
imagemagick is already the newest version (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
ls: cannot access 'media/textures/castpol01.png': No such file or directory


In [10]:
import os

# 1. Crea la cartella se non esiste
os.makedirs('/content/utils', exist_ok=True)

# 2. Scrivi il contenuto nel file utils/image_utils.py
with open('/content/utils/image_utils.py', 'w') as f:
    f.write('''
import os

import cv2
import flip_evaluator
import imageio.v3 as iio
import matplotlib
import matplotlib.font_manager as font_manager
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Ellipse
from numpy.linalg import norm
from PIL import Image
from scipy.ndimage import sobel

FONT_PATH = "assets/fonts/linux_libertine/LinLibertine_R.ttf"
font_manager.fontManager.addfont(FONT_PATH)
FONT_PROP = font_manager.FontProperties(fname=FONT_PATH).get_name()

plt.rcParams['font.family'] = FONT_PROP
plt.rcParams['text.usetex'] = True
matplotlib.rcParams['font.size'] = 16
matplotlib.rcParams['axes.titlesize'] = 16
matplotlib.rcParams['figure.titlesize'] = 16
matplotlib.rcParams['legend.fontsize'] = 16
matplotlib.rcParams['legend.title_fontsize'] = 16
matplotlib.rcParams['xtick.labelsize'] = 14
matplotlib.rcParams['ytick.labelsize'] = 14

ALLOWED_IMAGE_FILE_FORMATS = [".jpeg", ".jpg", ".png", ".tiff", ".exr"]

PLOT_DPI = 72.0
GAUSSIAN_ZOOM = 5
GAUSSIAN_COLOR = "#80ed99"


def get_psnr(image1, image2, mask, max_value=1.0):
    # Assicuriamoci che la maschera sia [1, H, W]
    if mask.dim() == 2:
        mask = mask.unsqueeze(0)
    if mask.shape[0] != 1:
        mask = mask.mean(dim=0, keepdim=True)

    # Consideriamo solo i pixel attivi nella maschera per evitare che NaN/inf fuori maschera
    # azzerata penalizzi il PSNR.
    active_mask = mask > 0.5
    if active_mask.sum() == 0:
        return 0.0

    # Espandiamo la maschera sui canali e selezioniamo solo i pixel validi
    if image1.dim() == 2:
        image1 = image1.unsqueeze(0)
    if image2.dim() == 2:
        image2 = image2.unsqueeze(0)
    active_mask = active_mask.expand_as(image1)
    active_pred = image1[active_mask]
    active_gt = image2[active_mask]

    if active_pred.numel() == 0:
        return 0.0
    if torch.isnan(active_pred).any() or torch.isinf(active_pred).any() or torch.isnan(active_gt).any() or torch.isinf(active_gt).any():
        return 0.0

    # Clamp per sicurezza, anche se dovrebbe essere già fatto upstream
    active_pred = torch.clamp(active_pred, 0.0, max_value)
    active_gt = torch.clamp(active_gt, 0.0, max_value)

    mse_masked = torch.mean((active_pred - active_gt) ** 2)
    if mse_masked.item() <= 1e-7:
        return float('inf')
    psnr_masked = 20 * torch.log10(torch.tensor(max_value, device=image1.device) / torch.sqrt(mse_masked))
    return psnr_masked.item()


def get_grid(h, w, x_lim=np.asarray([0, 1]), y_lim=np.asarray([0, 1])):
    x = torch.linspace(x_lim[0], x_lim[1], steps=w + 1)[:-1] + 0.5 / w
    y = torch.linspace(y_lim[0], y_lim[1], steps=h + 1)[:-1] + 0.5 / h
    grid_x, grid_y = torch.meshgrid(x, y, indexing='xy')
    grid = torch.stack([grid_x, grid_y], dim=-1)
    return grid


def compute_image_gradients(image):
    gy, gx = [], []
    for image_channel in image:
        gy.append(sobel(image_channel, 0))
        gx.append(sobel(image_channel, 1))
    gy = norm(np.stack(gy, axis=0), ord=2, axis=0).astype(np.float32)
    gx = norm(np.stack(gx, axis=0), ord=2, axis=0).astype(np.float32)
    return gy, gx


def load_images(load_path, downsample_ratio=None, gamma=None):
    """
    Load target images or textures from a directory or a single file.
    """
    image_list = []
    image_path_list = []
    image_fname_list = []
    num_channels_list = []
    bit_depth_list = []
    if os.path.isfile(load_path) and os.path.splitext(load_path)[1].lower() in ALLOWED_IMAGE_FILE_FORMATS:
        image_path_list.append(load_path)
    elif os.path.isdir(load_path):
        for file in sorted(os.listdir(load_path), key=str.lower):
            if os.path.splitext(file)[1].lower() in ALLOWED_IMAGE_FILE_FORMATS:
                image_path_list.append(os.path.join(load_path, file))
    if len(image_path_list) == 0:
        raise FileNotFoundError(f"No supported image file (JPEG, PNG, TIFF, EXR) found at '{load_path}'")
    for image_path in image_path_list:
        image_fname_list.append(os.path.splitext(os.path.basename(image_path))[0])
        # Warning: Only support image files in JPEG, PNG, TIFF, or EXR format
        image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
        if not len(image.shape) in [2, 3]:
            raise ValueError(f"Invalid image file ({os.path.basename(image_path)}) with shape {image.shape}")
        if len(image.shape) == 2:
            image = np.expand_dims(image, axis=2)
        else:
            if image.shape[-1] not in [1, 3, 4]:
                raise ValueError(f"Invalid image file ({os.path.basename(image_path)}) with shape {image.shape}")
            if image.shape[-1] == 4:
                image = image[..., :3]
            image = image[..., ::-1]
        num_channels = image.shape[-1]
        num_channels_list.append(num_channels)
        # 8 bit color depth
        if image.dtype == np.uint8:
            image = image.astype(np.float32) / 255.0
            bit_depth_list.append(8)
        # 16 bit color depth
        elif image.dtype == np.uint16:
            image = image.astype(np.float32) / 65535.0
            bit_depth_list.append(16)
        # 32 bit color depth
        else:
            if image.dtype != np.float32:
                raise ValueError(f"Unsupported image dtype: {image.dtype}")
            bit_depth_list.append(32)
        if downsample_ratio is not None:
            height, width = image.shape[:2]
            image = cv2.resize(image, (round(width/downsample_ratio), round(height/downsample_ratio)), interpolation=cv2.INTER_LANCZOS4)
        if gamma is not None:
            image = np.power(image, gamma)
        image = image.transpose(2, 0, 1)
        image_list.append(image)
    return np.concatenate(image_list, axis=0), num_channels_list, image_fname_list, bit_depth_list


def to_output_format(image, image_format, gamma):
    if image_format not in ALLOWED_IMAGE_FILE_FORMATS:
        raise ValueError(f"Invalid image format: {image_format}")
    if len(image.shape) not in [2, 3]:
        raise ValueError(f"Invalid image format: shape = {image.shape}")
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().clone().numpy()
    if len(image.shape) == 3 and image.shape[2] not in [1, 3]:
        image = image.transpose(1, 2, 0)
        if image.shape[2] not in [1, 3]:
            raise ValueError(f"Invalid image format: shape = {image.shape}")
    if len(image.shape) == 3 and image.shape[2] == 1:
        image = image.squeeze(axis=2)
    image = image.astype(np.float32)
    if gamma is not None:
        image = np.power(image, 1.0/gamma)
    if image_format in [".jpeg", ".jpg"]:
        image = np.clip(image, 0.0, 1.0)
        image = (255.0 * image).astype(np.uint8)
    elif image_format in [".png"]:
        image = np.clip(image, 0.0, 1.0)
        image = (65535.0 * image).astype(np.uint16)
    return image


def save_image(image, save_path, gamma=None, zoom=None):
    image_format = os.path.splitext(save_path)[1].lower()
    image = to_output_format(image, image_format, gamma)
    if zoom is not None and zoom > 0.0:
        height, width = image.shape[:2]
        image = cv2.resize(image, (round(width*zoom), round(height*zoom)), interpolation=cv2.INTER_NEAREST)
    if len(image.shape) == 3:
        image = image[..., ::-1]
    cv2.imwrite(save_path, image)


def separate_image_channels(images, input_channels):
    if len(images) != sum(input_channels):
        raise ValueError(f"Incompatible number of channels: {len(images):d} vs {sum(input_channels):d}")
    image_list = []
    curr_channel = 0
    for num_channels in input_channels:
        image_list.append(images[curr_channel:curr_channel+num_channels])
        curr_channel += num_channels
    return image_list


def visualize_gaussian_footprint(filepath, xy, scale, rot, feat, img_h, img_w, input_channels, alpha=0.8, gamma=None, save_image_format="jpg"):
    """
    Visualize the footprint of Gaussians using colored elliptical disks.
    """
    if feat.shape[1] != sum(input_channels):
        raise ValueError(f"Incompatible number of channels: {feat.shape[1]:d} vs {sum(input_channels):d}")
    xy = xy.detach().cpu().clone().numpy()
    y, x = xy[:, 1] * img_h, xy[:, 0] * img_w
    scale = GAUSSIAN_ZOOM * scale.detach().cpu().clone().numpy()
    rot = rot.detach().cpu().clone().numpy()
    if gamma is not None:
        feat = torch.pow(feat, 1.0/gamma)
    feat = np.clip(feat.detach().cpu().clone().numpy(), 0.0, 1.0)

    curr_channel = 0
    for image_id, num_channels in enumerate(input_channels, 1):
        curr_feat = feat[:, curr_channel:curr_channel+num_channels]
        if curr_feat.shape[1] == 1:
            curr_feat = np.repeat(curr_feat, 3, axis=1)
        fig = plt.figure()
        fig.set_dpi(PLOT_DPI)
        fig.set_size_inches(w=img_w/PLOT_DPI, h=img_h/PLOT_DPI, forward=False)
        ax = plt.gca()
        for gid in range(len(xy)):
            ellipse = Ellipse(xy=(x[gid], y[gid]), width=scale[gid, 0], height=scale[gid, 1],
                              angle=rot[gid, 0]*180/np.pi, alpha=alpha, ec=None, fc=curr_feat[gid], lw=None)
            ax.add_patch(ellipse)
        plt.xlim(0, img_w)
        plt.ylim(img_h, 0)
        plt.axis('off')
        plt.tight_layout()
        suffix = "" if len(input_channels) == 1 else f"_{image_id:d}"
        plt.savefig(f"{filepath}{suffix}.{save_image_format}", bbox_inches='tight', pad_inches=0, dpi=PLOT_DPI)
        plt.close()
        curr_channel += num_channels


def visualize_gaussian_position(filepath, images, xy, input_channels, color="#7bf1a8", size=700, every_n=10, alpha=0.8, gamma=None, save_image_format="jpg"):
    """
    Visualize the position of Gaussians using dots.
    """
    if len(images) != sum(input_channels):
        raise ValueError(f"Incompatible number of channels: {len(images):d} vs {sum(input_channels):d}")
    image_height, image_width = images.shape[1:]
    xy = xy.detach().cpu().clone().numpy()[::every_n]
    x, y = xy[:, 0] * image_width, xy[:, 1] * image_height

    curr_channel = 0
    for image_id, num_channels in enumerate(input_channels, 1):
        image = images[curr_channel:curr_channel+num_channels]
        image = to_output_format(image, f".{save_image_format}", gamma)
        fig = plt.figure()
        fig.set_dpi(PLOT_DPI)
        fig.set_size_inches(w=image_width/PLOT_DPI, h=image_height/PLOT_DPI, forward=False)
        plt.imshow(Image.fromarray(image), cmap='gray', vmin=0, vmax=255)
        plt.scatter(x, y, s=size, c=color, marker='o', alpha=alpha)
        plt.xlim(0, image_width)
        plt.ylim(image_height, 0)
        plt.axis('off')
        plt.tight_layout()
        suffix = "" if len(input_channels) == 1 else f"_{image_id:d}"
        plt.savefig(f"{filepath}{suffix}.{save_image_format}", bbox_inches='tight', pad_inches=0, dpi=PLOT_DPI)
        plt.close()
        curr_channel += num_channels


def visualize_added_gaussians(filepath, images, old_xy, new_xy, input_channels, size=500, every_n=5, alpha=0.8, gamma=None, save_image_format="jpg"):
    """
    Visualize the positions of added Gaussians during error-guided progressive optimization.
    """
    if len(images) != sum(input_channels):
        raise ValueError(f"Incompatible number of channels: {len(images):d} vs {sum(input_channels):d}")
    image_height, image_width = images.shape[1:]
    old_xy = old_xy.detach().cpu().clone().numpy()[::every_n]
    new_xy = new_xy.detach().cpu().clone().numpy()[::every_n]
    old_x, old_y = old_xy[:, 0] * image_width, old_xy[:, 1] * image_height
    new_x, new_y = new_xy[:, 0] * image_width, new_xy[:, 1] * image_height

    curr_channel = 0
    for image_id, num_channels in enumerate(input_channels, 1):
        image = images[curr_channel:curr_channel+num_channels]
        image = to_output_format(image, f".{save_image_format}", gamma)
        fig = plt.figure()
        fig.set_dpi(PLOT_DPI)
        fig.set_size_inches(w=image_width/PLOT_DPI, h=image_height/PLOT_DPI, forward=False)
        plt.imshow(Image.fromarray(image), cmap='gray', vmin=0, vmax=255)
        plt.scatter(old_x, old_y, s=size, c="#ef476f", marker='o', alpha=alpha)  # red
        plt.scatter(new_x, new_y, s=size, c="#06d6a0", marker='o', alpha=alpha)  # green
        plt.xlim(0, image_width)
        plt.ylim(image_height, 0)
        plt.axis('off')
        plt.tight_layout()
        suffix = "" if len(input_channels) == 1 else f"_{image_id:d}"
        plt.savefig(f"{filepath}{suffix}.{save_image_format}", bbox_inches='tight', pad_inches=0, dpi=PLOT_DPI)
        plt.close()
        curr_channel += num_channels


def save_error_maps(path, images, gt_images, channels, gamma, save_image_format="jpg"):
    images = torch.pow(torch.clamp(images, 0.0, 1.0), 1.0/gamma)
    gt_images = torch.pow(gt_images, 1.0/gamma)
    images_sep = separate_image_channels(images, channels)
    gt_images_sep = separate_image_channels(gt_images, channels)
    for idx, (image, gt_image) in enumerate(zip(images_sep, gt_images_sep), 1):
        gt_image, image = gt_image.detach().cpu().clone().numpy(), image.detach().cpu().clone().numpy()
        if gt_image.shape[0] == 1:
            gt_image = np.repeat(gt_image, 3, axis=0)
            image = np.repeat(image, 3, axis=0)
        gt_image, image = gt_image.transpose(1, 2, 0), image.transpose(1, 2, 0)
        suffix = "" if len(images_sep) == 1 else f"_{idx:d}"
        flip_error_map, _, _ = flip_evaluator.evaluate(reference=gt_image, test=image, dynamicRangeString="LDR", inputsRGB=True, applyMagma=True)
        save_image(flip_error_map, f"{path}{suffix}.{save_image_format}")

''')

print("✅ File /content/utils/image_utils.py aggiornato correttamente!")

✅ File /content/utils/image_utils.py aggiornato correttamente!


In [15]:
import time

for i in range (1, 1000):
  print(f"cycle: {i}")
  time.sleep(10)

cycle: 1
cycle: 2
cycle: 3
cycle: 4
cycle: 5
cycle: 6
cycle: 7
cycle: 8
cycle: 9
cycle: 10
cycle: 11
cycle: 12
cycle: 13
cycle: 14
cycle: 15
cycle: 16
cycle: 17
cycle: 18
cycle: 19
cycle: 20
cycle: 21
cycle: 22
cycle: 23
cycle: 24
cycle: 25
cycle: 26
cycle: 27
cycle: 28
cycle: 29
cycle: 30
cycle: 31
cycle: 32
cycle: 33
cycle: 34
cycle: 35
cycle: 36
cycle: 37
cycle: 38
cycle: 39
cycle: 40
cycle: 41
cycle: 42
cycle: 43
cycle: 44
cycle: 45
cycle: 46
cycle: 47
cycle: 48
cycle: 49
cycle: 50
cycle: 51
cycle: 52
cycle: 53
cycle: 54
cycle: 55
cycle: 56
cycle: 57
cycle: 58
cycle: 59
cycle: 60
cycle: 61
cycle: 62
cycle: 63
cycle: 64
cycle: 65
cycle: 66
cycle: 67
cycle: 68
cycle: 69
cycle: 70
cycle: 71
cycle: 72
cycle: 73
cycle: 74
cycle: 75
cycle: 76
cycle: 77
cycle: 78
cycle: 79
cycle: 80
cycle: 81
cycle: 82
cycle: 83
cycle: 84
cycle: 85
cycle: 86
cycle: 87
cycle: 88
cycle: 89
cycle: 90
cycle: 91
cycle: 92
cycle: 93
cycle: 94
cycle: 95
cycle: 96
cycle: 97
cycle: 98
cycle: 99
cycle: 100
cycle: 1

KeyboardInterrupt: 

In [ ]:
import os
import subprocess
import pandas as pd
import glob
from PIL import Image
import re

try:
    import wandb
    HAS_WANDB = True
except ImportError:
    HAS_WANDB = False

# --- CONFIGURAZIONE ---
csv_path = "baseline_gaussian_results.csv"
input_dir = "media/textures"
small_dir = "media/textures_small"
os.makedirs(small_dir, exist_ok=True)

# Ordiniamo le immagini per avere un avanzamento costante
images = sorted(glob.glob(f"{input_dir}/*.png"))[:30]
densities = [1000, 2500, 5000, 10000, 20000]
# Per la baseline gaussiana usiamo solo beta = 0
beta_val = 0

# --- 1. LOGIN E RESUME ---
completed = set()
results = []
run = None

if HAS_WANDB:
    try:
        # In Colab solitamente si fa il login manuale o tramite env
        wandb.login()
        run = wandb.init(project="gaussian-baseline-texture", name="baseline_marathon", resume=True)

        try:
            artifact = run.use_artifact('baseline_results:latest')
            artifact_dir = artifact.download()
            df_existing = pd.read_csv(os.path.join(artifact_dir, csv_path))
            completed = set(zip(df_existing['texture'], df_existing['gaussians']))
            results = df_existing.to_dict('records')
            print(f"Cloud Resume: {len(completed)} esperimenti saltati.")
        except:
            if os.path.exists(csv_path):
                df_local = pd.read_csv(csv_path)
                completed = set(zip(df_local['texture'], df_local['gaussians']))
                results = df_local.to_dict('records')
    except Exception as e:
        print(f"⚠️ Errore WandB: {e}")
        HAS_WANDB = False

# --- 2. LOOP ESPERIMENTI ---
for img_path in images:
    img_name = os.path.basename(img_path)
    small_path = os.path.join(small_dir, img_name)
    if not os.path.exists(small_path):
        with Image.open(img_path) as img:
            img = img.resize((1024, 1024), Image.LANCZOS)
            img.save(small_path)

    for n in densities:
        if (img_name, n) in completed:
            continue

        # Nome esperimento specifico per baseline
        exp_name = f"baseline_{img_name.split('.')[0]}_n{n}"
        print(f"\n>>> Running Baseline: {exp_name}")

        # Percorso corretto per Colab (verifica se devi aggiungere /content/)
        input_path_for_cmd = f"textures_small/{img_name}"

        cmd = [
            "python", "main.py",
            f"--input_path={input_path_for_cmd}",
            f"--exp_name={exp_name}",
            f"--num_gaussians={str(n)}",
            "--quantize",
            "--use_mask"
        ]

        try:
            # shell=True a volte aiuta in Colab se i path hanno spazi
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

            final_metrics = {}
            for line in process.stdout:
                print(line, end="")
                # Cerchiamo la riga finale con tutte le metriche
                if "PSNR:" in line and "LPIPS:" in line:
                    try:
                        # Regex per estrarre i valori dopo i due punti, ignorando i timestamp
                        psnr_match  = re.search(r'PSNR:\s*([\d.]+)', line)
                        ssim_match  = re.search(r'SSIM:\s*([\d.]+)', line)
                        lpips_match = re.search(r'LPIPS:\s*([\d.]+)', line)

                        if psnr_match and ssim_match and lpips_match:
                            final_metrics = {
                                "texture":   img_name,
                                "gaussians": n,
                                "beta":      beta_val, # Sarà sempre 0
                                "psnr":      float(psnr_match.group(1)),
                                "ssim":      float(ssim_match.group(1)),
                                "lpips":     float(lpips_match.group(1))
                            }
                    except Exception as parse_err:
                        print(f"Parse error: {parse_err}")
                        continue

            process.wait()

            if final_metrics:
                results.append(final_metrics)
                pd.DataFrame(results).to_csv(csv_path, index=False)

                if HAS_WANDB:
                    wandb.log(final_metrics)
                    if len(results) % 5 == 0:
                        new_art = wandb.Artifact('baseline_results', type='dataset')
                        new_art.add_file(csv_path)
                        run.log_artifact(new_art)

        except Exception as e:
            print(f"Errore critico su {exp_name}: {e}")
            continue
if HAS_WANDB:
    final_art = wandb.Artifact('baseline_results', type='dataset')
    final_art.add_file(csv_path)
    run.log_artifact(final_art)
    wandb.finish()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


wandb:   1 of 1 files downloaded.  


Cloud Resume: 40 esperimenti saltati.

>>> Running Baseline: baseline_castpol01_n1000
[2026/04/24 18:05:06] Start optimizing 1000 Gaussians for 'textures_small/castpol01.png'
[2026/04/24 18:05:06] ***********************************************
Maschera creata con Flood Fill. Pixel attivi: 793500/1048576
[2026/04/24 18:05:06] Spawn Mask: Utilizzo maschera UV precisa.
[2026/04/24 18:05:07] Uncompressed: 3145.73 KB | 24.000 bpp | 8.000 bppc
[2026/04/24 18:05:07] Compressed: 16.00 KB | 0.122 bpp | 0.041 bppc
[2026/04/24 18:05:07] Compression rate: 196.61x | 0.51%
[2026/04/24 18:05:07] ***********************************************
[2026/04/24 18:05:07] Image gradient map successfully saved
[2026/04/24 18:05:07] ***********************************************
[2026/04/24 18:05:12] Step: 100 | Total: 3.24 s | Render: 0.33 s | Loss: 0.1230, L1: 0.0483, SSIM: 0.0747 | PSNR: 20.42 | SSIM: 0.2532
[2026/04/24 18:05:15] Step: 200 | Total: 6.47 s | Render: 0.61 s | Loss: 0.1152, L1: 0.0412, SSIM:

In [ ]:
!python run_experiments.py